# TEMA + MACD-V + RSI Ensemble — Cell Fracture Edition

**Objective:** Build a 3-indicator ensemble that combines trend-following (TEMA), volatility-normalized momentum (MACD-V), and mean-reversion timing (RSI) into a unified signal using multiple voting modes.

**The Core Problem:** Single indicators are fragile. Triple EMA works in trends but whipsaws in ranges. RSI catches reversals but bleeds in trends. MACD-V measures momentum but is slow. **Ensembling reduces single-indicator failure modes.**

**Ensemble Voting Modes:**

| Mode | Logic | Philosophy | Risk Profile |
|------|-------|-----------|-------------|
| **AND** (Consensus) | All 3 must agree | Maximum conviction, fewest trades | Low frequency, high precision |
| **MAJORITY** (2-of-3) | Any 2 of 3 agree | Democratic vote, balanced | Moderate frequency |
| **OR** (Any-fires) | Any 1 is enough | Cast wide net, most trades | High frequency, lower precision |

**Sources:**
- Carver, *Systematic Trading* — forecast diversification multiplier, combining rules across styles (~0.25 correlation → multiplier ≈ 1.41 for 3 rules)
- Giordano (2018 Dow Award) — ATR Trend/Breakout, multi-factor Ranking Model
- SigTech (2022) — RSI as trend indicator alongside MACD
- Your existing ensemble engine (boruta.txt) — OR logic for signal combination

---


## 1. Environment & Dependencies

### What: Import all libraries.
### Why: One cell, zero ambiguity. If this cell fails, nothing else runs.
### Assumption: Libraries pre-installed. If not, uncomment the pip line.


In [ ]:
# !pip install yfinance TA-Lib vectorbt scipy pandas numpy matplotlib --quiet

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 140)

print("✅ All imports loaded")
print(f"   vectorbt: {vbt.__version__}  |  pandas: {pd.__version__}  |  numpy: {np.__version__}")


## 2. Configuration — Single Source of Truth

### What: Every tunable parameter lives here. Change the asset, date range, or grid — only this cell.
### Why: Prevents magic numbers scattered through 50+ cells. One cell to rule them all.
### Assumption: BTC-USD = 365-day calendar. For equity/FX tickers, set `ANNUAL_FACTOR = 252`.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# MASTER CONFIGURATION — EDIT ONLY THIS CELL
# ═══════════════════════════════════════════════════════════════

TICKER         = "BTC-USD"        # Asset under test
START_DATE     = "2018-01-01"     # Download start
END_DATE       = None             # None = latest available
TRAIN_RATIO    = 0.70             # 70% train, 30% OOS
ANNUAL_FACTOR  = 365              # 365 for crypto, 252 for equities

# ─── TEMA (Triple EMA) Parameter Grid ───
EMA1_RANGE     = range(5, 22, 2)     # Fast EMA   (5,7,9,...,21)
EMA2_RANGE     = range(20, 55, 5)    # Medium EMA (20,25,...,50)
EMA3_RANGE     = range(50, 130, 10)  # Slow EMA   (50,60,...,120)

# ─── MACD-V Parameter Grid ───
MACDV_FAST     = range(8, 16, 2)     # MACD fast EMA
MACDV_SLOW     = range(20, 35, 3)    # MACD slow EMA
MACDV_SIGNAL   = range(5, 12, 2)     # Signal line EMA
ATR_LEN        = 14                  # ATR for volatility normalization

# ─── RSI Parameter Grid ───
RSI_LEN_RANGE      = range(8, 22, 2)       # RSI lookback period
RSI_OVERSOLD_RANGE = range(25, 40, 5)      # Oversold threshold (entry)
RSI_OVERBOUGHT_RANGE = range(65, 80, 5)    # Overbought threshold (exit)

# ─── Regime & Volatility Filters ───
SMA_REGIME_LEN     = 200
VOL_LOOKBACK       = 20
VOL_PERCENTILE_LO  = 5
VOL_PERCENTILE_HI  = 99

# ─── Ensemble Settings ───
ENSEMBLE_MODES     = ['AND', 'MAJORITY', 'OR']  # Voting modes to test
MIN_TRADES         = 10                          # Minimum trades for valid Sharpe

# ─── Execution ───
SIGNAL_SHIFT       = 1              # Anti-lookahead bar shift
FEES               = 0.001          # 0.1% per trade (one-way)
INIT_CASH          = 100_000

# ─── Sensitivity ───
SENSITIVITY_STEPS  = 5              # +/- N steps around optimum

print("✅ Configuration loaded")
print(f"   Asset: {TICKER} | Dates: {START_DATE} → {END_DATE or 'today'}")
print(f"   Split: {TRAIN_RATIO:.0%} train / {1-TRAIN_RATIO:.0%} OOS")
print(f"   Ensemble modes: {ENSEMBLE_MODES}")
print(f"   TEMA grid:  {len(list(EMA1_RANGE))}×{len(list(EMA2_RANGE))}×{len(list(EMA3_RANGE))} raw combos")
print(f"   MACDV grid: {len(list(MACDV_FAST))}×{len(list(MACDV_SLOW))}×{len(list(MACDV_SIGNAL))} raw combos")
print(f"   RSI grid:   {len(list(RSI_LEN_RANGE))}×{len(list(RSI_OVERSOLD_RANGE))}×{len(list(RSI_OVERBOUGHT_RANGE))} raw combos")
print(f"   Signal shift: {SIGNAL_SHIFT} bar(s) anti-lookahead")


## 3. Data Download & Multi-Index Handling

### What: Download OHLCV via yfinance and flatten the MultiIndex columns.
### Why: yfinance returns `(Price, Ticker)` MultiIndex which silently breaks TA-Lib and vectorbt.
### Assumption: Single-ticker download.


In [ ]:
raw_df = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Flatten MultiIndex
if isinstance(raw_df.columns, pd.MultiIndex):
    print("⚠️  MultiIndex detected — flattening")
    df = raw_df.droplevel('Ticker', axis=1)
else:
    df = raw_df.copy()

required = ['Open', 'High', 'Low', 'Close', 'Volume']
missing = [c for c in required if c not in df.columns]
assert len(missing) == 0, f"❌ Missing columns: {missing}"

print(f"\n📥 {TICKER}: {len(df)} rows | {df.index[0].date()} → {df.index[-1].date()}")
print(f"   Columns: {list(df.columns)} | dtypes all float: {all(df[required[:4]].dtypes == 'float64')}")


## 4. Data Sanity Audit

### What: Verify monotonic timestamps, NaN counts, OHLC integrity.
### Why: Garbage in → garbage out. Every indicator downstream trusts this data.
### Assumption: Daily bars, no intraday mixing.


In [ ]:
# 4a. Index monotonicity
assert df.index.is_monotonic_increasing, "❌ Index NOT time-ordered!"
assert df.index.is_unique, "❌ Duplicate dates found!"
print("✅ Index: monotonic & unique")

# 4b. NaN audit
nan_pre = df.isna().sum().sum()
df = df.ffill().bfill()
nan_post = df.isna().sum().sum()
print(f"✅ NaNs: {nan_pre} before fill → {nan_post} after fill")

# 4c. OHLC integrity
close = df['Close'].astype(float)
high  = df['High'].astype(float)
low   = df['Low'].astype(float)
volume = df['Volume'].astype(float)

violations = ((high < close) | (close < low)).sum()
print(f"✅ OHLC integrity: {len(close)-violations}/{len(close)} bars pass (H≥C≥L)")
print(f"   Shape: {df.shape} | Range: {close.iloc[0]:.2f} → {close.iloc[-1]:.2f}")


## 5. Shared Components — ATR & Regime/Volatility Filters

### What: Compute ATR (shared volatility ruler), 200-SMA regime filter, and volatility percentile filter.
### Why: These are shared across ALL three indicators. Compute once, apply everywhere.
### Assumption: ATR_LEN=14 (Wilder default). SMA_REGIME_LEN=200 (industry standard).


In [ ]:
# 5a. ATR — the "invariant ruler"
atr = pd.Series(talib.ATR(high.values, low.values, close.values, timeperiod=ATR_LEN), index=close.index)
print(f"✅ ATR({ATR_LEN}): mean={atr.mean():.2f}, NaN warmup={atr.isna().sum()}")


In [ ]:
# 5b. Regime filter (200-SMA)
sma_200 = pd.Series(talib.SMA(close.values, timeperiod=SMA_REGIME_LEN), index=close.index)
regime_bull = close > sma_200
print(f"✅ Regime SMA-{SMA_REGIME_LEN}: {regime_bull.sum()} bullish bars ({regime_bull.mean():.1%})")
print(f"   Current: {'BULL ✅' if regime_bull.iloc[-1] else 'BEAR ❌'}")


In [ ]:
# 5c. Volatility filter (rolling ATR percentile)
atr_pctile = atr.rolling(252).apply(lambda x: stats.percentileofscore(x, x.iloc[-1]), raw=False)
vol_ok = ((atr_pctile > VOL_PERCENTILE_LO) & (atr_pctile < VOL_PERCENTILE_HI)).fillna(False)
print(f"✅ Vol filter: {vol_ok.sum()} tradeable bars ({vol_ok.mean():.1%})")
print(f"   Excluded (dead market <{VOL_PERCENTILE_LO}%): {(atr_pctile <= VOL_PERCENTILE_LO).sum()}")
print(f"   Excluded (extreme >{VOL_PERCENTILE_HI}%): {(atr_pctile >= VOL_PERCENTILE_HI).sum()}")


## 6. Indicator Signal Generators — The Three Lenses

Each indicator measures the market through a different lens:

| Indicator | Lens | Strength | Weakness |
|-----------|------|----------|----------|
| **TEMA** | Trend alignment (EMA hierarchy) | Strong in sustained trends | Whipsaws in choppy ranges |
| **MACD-V** | Volatility-normalized momentum | Comparable across regimes | Lags at turning points |
| **RSI** | Mean-reversion momentum | Catches oversold bounces | Bleeds in persistent trends |

**The ensemble hypothesis:** By combining all three, we reduce each indicator's individual weakness. When TEMA says "trend is up", MACD-V says "momentum is strong relative to vol", AND RSI says "not overbought", conviction is high.


### 6a. TEMA Signal Generator (Trend Direction)

### What: Triple EMA crossover — entry when EMA1 > EMA2 > EMA3 (aligned bullish), exit when fast < medium.
### Why: Captures trend alignment. The "stacking" of three EMAs provides a stronger filter than a simple 2-EMA cross.
### Assumption: Returns raw boolean signals — NOT shifted. Shift happens at ensemble level.


In [ ]:
def compute_tema_signals(close_s, e1, e2, e3):
    """Triple EMA: Entry = alignment (fast>med>slow), Exit = fast<med.
    Returns (entries, exits) as boolean Series — NOT shifted."""
    ema1 = pd.Series(talib.EMA(close_s.values, timeperiod=e1), index=close_s.index)
    ema2 = pd.Series(talib.EMA(close_s.values, timeperiod=e2), index=close_s.index)
    ema3 = pd.Series(talib.EMA(close_s.values, timeperiod=e3), index=close_s.index)
    
    bullish = (ema1 > ema2) & (ema2 > ema3)
    entries = bullish & ~bullish.shift(1).fillna(False)
    exits   = (ema1 < ema2) & (ema1.shift(1) >= ema2.shift(1))
    return entries, exits

# Quick sanity check
_te, _tx = compute_tema_signals(close, 10, 30, 70)
print(f"✅ TEMA generator defined — demo (10/30/70): {_te.sum()} entries, {_tx.sum()} exits")


### 6b. MACD-V Signal Generator (Volatility-Normalized Momentum)

### What: MACD line divided by ATR. Entry = MACD-V crosses above signal line. Exit = crosses below.
### Why: Raw MACD is not comparable across assets or volatility regimes. Dividing by ATR creates Carver's "invariant ruler" — a normalized momentum score where ±1.0 = one ATR unit of momentum.
### Assumption: Uses the pre-computed ATR from Section 5. Signal line is an EMA of MACD-V.


In [ ]:
def compute_macdv_signals(close_s, atr_s, fast, slow, signal):
    """MACD-V: Volatility-normalized MACD. Entry/exit on signal line cross.
    Returns (entries, exits) as boolean Series — NOT shifted."""
    macd_line = pd.Series(
        talib.EMA(close_s.values, timeperiod=fast) - talib.EMA(close_s.values, timeperiod=slow),
        index=close_s.index
    )
    macd_v = macd_line / atr_s.replace(0, np.nan)
    sig_line = pd.Series(talib.EMA(macd_v.values, timeperiod=signal), index=close_s.index)
    
    entries = (macd_v > sig_line) & (macd_v.shift(1) <= sig_line.shift(1))
    exits   = (macd_v < sig_line) & (macd_v.shift(1) >= sig_line.shift(1))
    return entries, exits

_me, _mx = compute_macdv_signals(close, atr, 12, 26, 9)
print(f"✅ MACD-V generator defined — demo (12/26/9): {_me.sum()} entries, {_mx.sum()} exits")


### 6c. RSI Signal Generator (Mean-Reversion Timing)

### What: RSI crossing up from oversold zone = entry. RSI crossing down from overbought = exit.
### Why: RSI adds a contrarian timing element. In the ensemble, it prevents entries when momentum is already exhausted (overbought) and catches bounce entries that pure trend indicators miss.
### Assumption: Standard RSI (Wilder smoothing via TA-Lib). Oversold < 30, Overbought > 70 as defaults.

> **Skeptic Note:** RSI as a standalone mean-reversion signal is dangerous in trending markets. That's why it's ONLY used inside the ensemble, never alone.


In [ ]:
def compute_rsi_signals(close_s, rsi_len, oversold, overbought):
    """RSI: Entry = crosses up from oversold. Exit = crosses down from overbought.
    Returns (entries, exits) as boolean Series — NOT shifted."""
    rsi = pd.Series(talib.RSI(close_s.values, timeperiod=rsi_len), index=close_s.index)
    
    # Entry: RSI crosses up through oversold level (recovery from oversold)
    entries = (rsi > oversold) & (rsi.shift(1) <= oversold)
    
    # Exit: RSI crosses down through overbought level (exhaustion)
    exits = (rsi < overbought) & (rsi.shift(1) >= overbought)
    
    return entries, exits

_re, _rx = compute_rsi_signals(close, 14, 30, 70)
print(f"✅ RSI generator defined — demo (14, OS=30, OB=70): {_re.sum()} entries, {_rx.sum()} exits")


## 7. Ensemble Combiner — The Voting Machine

### What: Combine entry/exit signals from all 3 indicators using AND, MAJORITY, or OR logic.
### Why: This is the core innovation. Per Carver, combining rules from different "styles" (trend vs momentum vs mean-reversion) has low correlation (~0.25), yielding a diversification multiplier of ~1.41 — the ensemble is more than the sum of its parts.

| Mode | Entry Logic | Exit Logic | Expected Behavior |
|------|-------------|------------|-------------------|
| AND | All 3 indicators must fire entry | ANY 1 fires exit | Very selective, few trades, high conviction |
| MAJORITY | 2 of 3 must fire entry | ANY 1 fires exit (to be safe) | Balanced trade frequency |
| OR | Any 1 fires entry | ALL 3 must fire exit (hold longer) | More trades, wider net |

### Assumption: Exit logic is asymmetric — we're more aggressive exiting (safety) than entering.

> **Design Choice:** For exits, AND mode uses `any` (exit if any indicator says sell), while OR mode uses `any` too (practical safety — don't wait for all 3 to agree to exit). The MAJORITY exit requires 2-of-3 for a balanced approach.


In [ ]:
def ensemble_signals(tema_ent, tema_exi, macdv_ent, macdv_exi, rsi_ent, rsi_exi, mode='MAJORITY'):
    """Combine 3 indicator signals using voting logic.
    
    Args:
        mode: 'AND' (all 3), 'MAJORITY' (2 of 3), 'OR' (any 1)
    
    Returns: (combined_entries, combined_exits) as boolean Series
    """
    # Stack entries as a DataFrame for vote counting
    ent_stack = pd.concat([
        tema_ent.rename('tema'), macdv_ent.rename('macdv'), rsi_ent.rename('rsi')
    ], axis=1).fillna(False)
    
    exi_stack = pd.concat([
        tema_exi.rename('tema'), macdv_exi.rename('macdv'), rsi_exi.rename('rsi')
    ], axis=1).fillna(False)
    
    entry_votes = ent_stack.sum(axis=1)  # 0, 1, 2, or 3
    exit_votes  = exi_stack.sum(axis=1)
    
    if mode == 'AND':
        # ALL 3 must agree to enter, ANY 1 to exit (conservative)
        combined_ent = entry_votes >= 3
        combined_exi = exit_votes >= 1
    elif mode == 'MAJORITY':
        # 2 of 3 to enter, 2 of 3 to exit (balanced)
        combined_ent = entry_votes >= 2
        combined_exi = exit_votes >= 2
    elif mode == 'OR':
        # ANY 1 to enter, 2 of 3 to exit (aggressive entry, balanced exit)
        combined_ent = entry_votes >= 1
        combined_exi = exit_votes >= 2
    else:
        raise ValueError(f"Unknown mode: {mode}")
    
    return combined_ent.astype(bool), combined_exi.astype(bool)

# Demo
_ce, _cx = ensemble_signals(_te, _tx, _me, _mx, _re, _rx, 'MAJORITY')
print(f"✅ Ensemble combiner defined")
print(f"   Demo MAJORITY (2-of-3): {_ce.sum()} entries, {_cx.sum()} exits")

for m in ENSEMBLE_MODES:
    ce, cx = ensemble_signals(_te, _tx, _me, _mx, _re, _rx, m)
    print(f"   {m:>8}: {ce.sum():>4} entries, {cx.sum():>4} exits")


## 8. Time-Ordered Train / OOS Split

### What: Split all data strictly by time. First 70% = training. Last 30% = untouched OOS.
### Why: Per Carver — "In sample testing should be avoided as it will produce extremely optimistic back-test results." No random splits, no peeking.
### Assumption: The split applies to ALL series (close, high, low, ATR, filters).


In [ ]:
split_idx  = int(len(df) * TRAIN_RATIO)
split_date = df.index[split_idx]

# Package train/test data for clean function passing
train_data = {
    'close': close.iloc[:split_idx], 'high': high.iloc[:split_idx],
    'low': low.iloc[:split_idx], 'atr': atr.iloc[:split_idx],
    'regime': regime_bull.iloc[:split_idx], 'vol_ok': vol_ok.iloc[:split_idx],
}
test_data = {
    'close': close.iloc[split_idx:], 'high': high.iloc[split_idx:],
    'low': low.iloc[split_idx:], 'atr': atr.iloc[split_idx:],
    'regime': regime_bull.iloc[split_idx:], 'vol_ok': vol_ok.iloc[split_idx:],
}

print(f"✅ Split at {split_date.date()} (index {split_idx})")
print(f"   TRAIN: {len(train_data['close'])} bars | {train_data['close'].index[0].date()} → {train_data['close'].index[-1].date()}")
print(f"   OOS:   {len(test_data['close'])} bars  | {test_data['close'].index[0].date()} → {test_data['close'].index[-1].date()}")
print(f"   ⚠️  OOS is SEALED until Section 12.")


## 9. Phase 1 — Optimize Each Indicator Individually on Training Data

### What: Find the best parameters for TEMA, MACD-V, and RSI independently.
### Why: We need each indicator "tuned to its best self" before combining them. Optimizing the ensemble directly over all 3 parameter grids would be computationally explosive and prone to overfitting.
### Assumption: Sharpe Ratio is the objective. Signals are shifted by SIGNAL_SHIFT bars. Regime + vol filters applied.

> **Carver's insight:** "Each rule variation should have a positive forecast weight." We optimize each indicator to its best version, then the ensemble voting mode decides how to combine them.


### 9a. TEMA Optimization


In [ ]:
print("🔄 Optimizing TEMA on training data...\n")

tema_results = []
combos = [(e1,e2,e3) for e1 in EMA1_RANGE for e2 in EMA2_RANGE for e3 in EMA3_RANGE if e1 < e2 < e3]
print(f"   Valid combos: {len(combos)}")

for (e1, e2, e3) in combos:
    try:
        ent, exi = compute_tema_signals(train_data['close'], e1, e2, e3)
        ent = ent & train_data['regime'] & train_data['vol_ok']
        ent = ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        exi = exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        if ent.sum() < MIN_TRADES: continue
        pf = vbt.Portfolio.from_signals(train_data['close'], entries=ent, exits=exi,
                                         init_cash=INIT_CASH, fees=FEES, freq='D')
        sr = pf.sharpe_ratio()
        if np.isnan(sr): continue
        tema_results.append({'e1':e1, 'e2':e2, 'e3':e3, 'sharpe':float(sr),
                             'ret':float(pf.total_return()), 'dd':float(pf.max_drawdown()),
                             'trades':int(pf.trades.count())})
    except: pass

tema_df = pd.DataFrame(tema_results).sort_values('sharpe', ascending=False)
best_tema = tema_df.iloc[0]
BEST_E1, BEST_E2, BEST_E3 = int(best_tema['e1']), int(best_tema['e2']), int(best_tema['e3'])

print(f"\n🏆 BEST TEMA: ({BEST_E1}, {BEST_E2}, {BEST_E3})")
print(f"   Sharpe={best_tema['sharpe']:.4f} | Return={best_tema['ret']:.2%} | DD={best_tema['dd']:.2%} | Trades={best_tema['trades']}")
print(f"   Tested: {len(tema_results)} valid combos")


### 9b. MACD-V Optimization


In [ ]:
print("🔄 Optimizing MACD-V on training data...\n")

macdv_results = []
combos = [(f,s,sg) for f in MACDV_FAST for s in MACDV_SLOW for sg in MACDV_SIGNAL if f < s]
print(f"   Valid combos: {len(combos)}")

for (f, s, sg) in combos:
    try:
        ent, exi = compute_macdv_signals(train_data['close'], train_data['atr'], f, s, sg)
        ent = ent & train_data['regime'] & train_data['vol_ok']
        ent = ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        exi = exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        if ent.sum() < MIN_TRADES: continue
        pf = vbt.Portfolio.from_signals(train_data['close'], entries=ent, exits=exi,
                                         init_cash=INIT_CASH, fees=FEES, freq='D')
        sr = pf.sharpe_ratio()
        if np.isnan(sr): continue
        macdv_results.append({'fast':f, 'slow':s, 'signal':sg, 'sharpe':float(sr),
                              'ret':float(pf.total_return()), 'dd':float(pf.max_drawdown()),
                              'trades':int(pf.trades.count())})
    except: pass

macdv_df = pd.DataFrame(macdv_results).sort_values('sharpe', ascending=False)
best_mv = macdv_df.iloc[0]
BEST_MF, BEST_MS, BEST_MSG = int(best_mv['fast']), int(best_mv['slow']), int(best_mv['signal'])

print(f"\n🏆 BEST MACD-V: ({BEST_MF}, {BEST_MS}, {BEST_MSG})")
print(f"   Sharpe={best_mv['sharpe']:.4f} | Return={best_mv['ret']:.2%} | DD={best_mv['dd']:.2%} | Trades={best_mv['trades']}")
print(f"   Tested: {len(macdv_results)} valid combos")


### 9c. RSI Optimization


In [ ]:
print("🔄 Optimizing RSI on training data...\n")

rsi_results = []
combos = [(r, os, ob) for r in RSI_LEN_RANGE for os in RSI_OVERSOLD_RANGE for ob in RSI_OVERBOUGHT_RANGE if os < ob]
print(f"   Valid combos: {len(combos)}")

for (r, os_val, ob_val) in combos:
    try:
        ent, exi = compute_rsi_signals(train_data['close'], r, os_val, ob_val)
        ent = ent & train_data['regime'] & train_data['vol_ok']
        ent = ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        exi = exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        if ent.sum() < MIN_TRADES: continue
        pf = vbt.Portfolio.from_signals(train_data['close'], entries=ent, exits=exi,
                                         init_cash=INIT_CASH, fees=FEES, freq='D')
        sr = pf.sharpe_ratio()
        if np.isnan(sr): continue
        rsi_results.append({'rsi_len':r, 'oversold':os_val, 'overbought':ob_val, 'sharpe':float(sr),
                            'ret':float(pf.total_return()), 'dd':float(pf.max_drawdown()),
                            'trades':int(pf.trades.count())})
    except: pass

rsi_df = pd.DataFrame(rsi_results).sort_values('sharpe', ascending=False)
best_rsi = rsi_df.iloc[0]
BEST_RL, BEST_OS, BEST_OB = int(best_rsi['rsi_len']), int(best_rsi['oversold']), int(best_rsi['overbought'])

print(f"\n🏆 BEST RSI: (len={BEST_RL}, OS={BEST_OS}, OB={BEST_OB})")
print(f"   Sharpe={best_rsi['sharpe']:.4f} | Return={best_rsi['ret']:.2%} | DD={best_rsi['dd']:.2%} | Trades={best_rsi['trades']}")
print(f"   Tested: {len(rsi_results)} valid combos")


### 9d. Phase 1 Summary — Best Individual Parameters


In [ ]:
print("\n" + "="*80)
print("📋 PHASE 1 SUMMARY — BEST INDIVIDUAL PARAMETERS (In-Sample)")
print("="*80)
summary_ind = pd.DataFrame([
    {'Indicator': 'TEMA', 'Params': f'({BEST_E1},{BEST_E2},{BEST_E3})', 
     'Sharpe': f"{best_tema['sharpe']:.4f}", 'Return': f"{best_tema['ret']:.2%}",
     'MaxDD': f"{best_tema['dd']:.2%}", 'Trades': int(best_tema['trades'])},
    {'Indicator': 'MACD-V', 'Params': f'({BEST_MF},{BEST_MS},{BEST_MSG})',
     'Sharpe': f"{best_mv['sharpe']:.4f}", 'Return': f"{best_mv['ret']:.2%}",
     'MaxDD': f"{best_mv['dd']:.2%}", 'Trades': int(best_mv['trades'])},
    {'Indicator': 'RSI', 'Params': f'(len={BEST_RL},OS={BEST_OS},OB={BEST_OB})',
     'Sharpe': f"{best_rsi['sharpe']:.4f}", 'Return': f"{best_rsi['ret']:.2%}",
     'MaxDD': f"{best_rsi['dd']:.2%}", 'Trades': int(best_rsi['trades'])},
])
print(summary_ind.to_string(index=False))
print("\n   ⚠️  These are INDIVIDUAL results. The ensemble may be better or worse.")
print("   ⚠️  All values are IN-SAMPLE. Real test = Section 12 (OOS).")


## 10. Phase 2 — Ensemble Voting Mode Optimization

### What: Using the best parameters for each indicator (frozen from Phase 1), test all three voting modes (AND, MAJORITY, OR) to find which combination method produces the best risk-adjusted performance.
### Why: The parameters are fixed — now we're optimizing HOW to combine them. This is a 3-option search, not a grid explosion. Per Carver, the key is the diversification benefit from combining low-correlation rules.
### Assumption: Parameters are FROZEN from Phase 1. Only the voting mode is tested.


In [ ]:
def run_ensemble_backtest(data, e1, e2, e3, mf, ms, msg, rl, os_v, ob_v, mode, shift=SIGNAL_SHIFT):
    """Full ensemble backtest with given params and voting mode.
    Returns: vbt.Portfolio object or None."""
    c, a, reg, vol = data['close'], data['atr'], data['regime'], data['vol_ok']
    
    # Generate raw signals from each indicator
    t_ent, t_exi = compute_tema_signals(c, e1, e2, e3)
    m_ent, m_exi = compute_macdv_signals(c, a, mf, ms, msg)
    r_ent, r_exi = compute_rsi_signals(c, rl, os_v, ob_v)
    
    # Combine via ensemble voting
    ens_ent, ens_exi = ensemble_signals(t_ent, t_exi, m_ent, m_exi, r_ent, r_exi, mode)
    
    # Apply regime + vol filters (on entries only)
    ens_ent = ens_ent & reg & vol
    
    # Anti-lookahead shift
    ens_ent = ens_ent.shift(shift).fillna(False).astype(bool)
    ens_exi = ens_exi.shift(shift).fillna(False).astype(bool)
    
    if ens_ent.sum() < 3:
        return None
    
    pf = vbt.Portfolio.from_signals(c, entries=ens_ent, exits=ens_exi,
                                     init_cash=INIT_CASH, fees=FEES, freq='D')
    return pf

# Test all voting modes on TRAINING data
print("🔄 Testing ensemble voting modes on training data...\n")

mode_results = {}
for mode in ENSEMBLE_MODES:
    pf = run_ensemble_backtest(train_data, BEST_E1, BEST_E2, BEST_E3,
                                BEST_MF, BEST_MS, BEST_MSG,
                                BEST_RL, BEST_OS, BEST_OB, mode)
    if pf is not None:
        sr = float(pf.sharpe_ratio())
        mode_results[mode] = {
            'pf': pf, 'sharpe': sr,
            'sortino': float(pf.sortino_ratio()),
            'ret': float(pf.total_return()),
            'dd': float(pf.max_drawdown()),
            'trades': int(pf.trades.count()),
            'win_rate': float(pf.trades.win_rate()) if pf.trades.count() > 0 else 0,
        }
        print(f"   {mode:>8}: Sharpe={sr:.4f} | Return={mode_results[mode]['ret']:.2%} | "
              f"DD={mode_results[mode]['dd']:.2%} | Trades={mode_results[mode]['trades']} | "
              f"WinRate={mode_results[mode]['win_rate']:.1%}")
    else:
        print(f"   {mode:>8}: ❌ Too few signals")

# Select best voting mode
BEST_MODE = max(mode_results, key=lambda k: mode_results[k]['sharpe'])
print(f"\n🏆 BEST ENSEMBLE MODE: {BEST_MODE}")
print(f"   Sharpe: {mode_results[BEST_MODE]['sharpe']:.4f}")


## 11. In-Sample Ensemble Backtest — Full Equity Curve

### What: Run the best ensemble (best voting mode + best params) on training data. Plot equity + drawdown.
### Why: Visual inspection catches single-trade dominance, suspicious flatlines, regime clustering.
### Assumption: This is STILL in-sample. Don't get excited yet.


In [ ]:
pf_is = mode_results[BEST_MODE]['pf']

print(f"✅ In-Sample Ensemble Backtest — {BEST_MODE} mode")
print(f"   TEMA({BEST_E1},{BEST_E2},{BEST_E3}) + MACD-V({BEST_MF},{BEST_MS},{BEST_MSG}) + RSI({BEST_RL},{BEST_OS},{BEST_OB})")
print(f"{'='*65}")
print(f"   Sharpe:    {pf_is.sharpe_ratio():.4f}")
print(f"   Sortino:   {pf_is.sortino_ratio():.4f}")
print(f"   Return:    {pf_is.total_return():.2%}")
print(f"   Max DD:    {pf_is.max_drawdown():.2%}")
print(f"   Trades:    {pf_is.trades.count()}")
wr = pf_is.trades.win_rate() if pf_is.trades.count() > 0 else 0
print(f"   Win Rate:  {wr:.2%}")
pf_factor = pf_is.trades.profit_factor() if pf_is.trades.count() > 0 else 0
print(f"   Profit Factor: {pf_factor:.2f}")


In [ ]:
# Plot IS equity curve + drawdown
fig, axes = plt.subplots(2, 1, figsize=(15, 8), gridspec_kw={'height_ratios': [3, 1]})
title = f'In-Sample Ensemble [{BEST_MODE}] — TEMA({BEST_E1},{BEST_E2},{BEST_E3}) + MACD-V({BEST_MF},{BEST_MS},{BEST_MSG}) + RSI({BEST_RL},{BEST_OS},{BEST_OB})'
fig.suptitle(title, fontsize=12, fontweight='bold')

pf_is.value().plot(ax=axes[0], color='navy', linewidth=1.5)
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title(f'Sharpe: {pf_is.sharpe_ratio():.3f} | Return: {pf_is.total_return():.1%} | MaxDD: {pf_is.max_drawdown():.1%} | Trades: {pf_is.trades.count()}')
axes[0].grid(alpha=0.3)

pf_is.drawdown().plot(ax=axes[1], color='crimson', linewidth=1)
axes[1].fill_between(pf_is.drawdown().index, pf_is.drawdown().values, alpha=0.3, color='crimson')
axes[1].set_ylabel('Drawdown'); axes[1].set_xlabel('Date'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 12. Out-of-Sample Validation — The Moment of Truth

### What: Apply the EXACT same ensemble (frozen params + frozen voting mode) to the untouched OOS data.
### Why: THIS is the only honest measure. Per Carver: "Use the first half to fit… then test on the second out of sample period."
### Assumption: NOTHING changes. Same params, same mode, same filters, same shift.

> **Critical:** If OOS Sharpe degrades >50%, the ensemble is overfit. If OOS Sharpe > IS Sharpe, suspect data leak.


In [ ]:
pf_oos = run_ensemble_backtest(test_data, BEST_E1, BEST_E2, BEST_E3,
                                BEST_MF, BEST_MS, BEST_MSG,
                                BEST_RL, BEST_OS, BEST_OB, BEST_MODE)

print(f"✅ Out-of-Sample Ensemble Backtest — {BEST_MODE} mode")
print(f"{'='*65}")
if pf_oos is not None:
    print(f"   Sharpe:    {pf_oos.sharpe_ratio():.4f}")
    print(f"   Sortino:   {pf_oos.sortino_ratio():.4f}")
    print(f"   Return:    {pf_oos.total_return():.2%}")
    print(f"   Max DD:    {pf_oos.max_drawdown():.2%}")
    print(f"   Trades:    {pf_oos.trades.count()}")
    oos_wr = pf_oos.trades.win_rate() if pf_oos.trades.count() > 0 else 0
    print(f"   Win Rate:  {oos_wr:.2%}")
    oos_pf = pf_oos.trades.profit_factor() if pf_oos.trades.count() > 0 else 0
    print(f"   Profit Factor: {oos_pf:.2f}")
else:
    print("   ❌ Too few OOS signals — ensemble may be too restrictive for this regime")


### 12a. Comprehensive IS vs OOS Comparison


In [ ]:
is_sr  = float(pf_is.sharpe_ratio())
oos_sr = float(pf_oos.sharpe_ratio()) if pf_oos else float('nan')
degrad = (1 - oos_sr/is_sr)*100 if (is_sr != 0 and not np.isnan(oos_sr)) else float('nan')

def safe_metric(pf, fn_name, default='N/A'):
    if pf is None: return default
    try:
        val = getattr(pf, fn_name)()
        return f"{val:.4f}" if abs(val) < 100 else f"{val:.2f}"
    except: return default

def safe_pct(pf, fn_name, default='N/A'):
    if pf is None: return default
    try: return f"{getattr(pf, fn_name)():.2%}"
    except: return default

comparison = pd.DataFrame({
    'Metric': ['Sharpe Ratio', 'Sortino Ratio', 'Total Return', 'Max Drawdown',
               'Trades', 'Win Rate', 'Profit Factor'],
    'In-Sample': [
        f"{is_sr:.4f}", safe_metric(pf_is, 'sortino_ratio'),
        safe_pct(pf_is, 'total_return'), safe_pct(pf_is, 'max_drawdown'),
        str(pf_is.trades.count()),
        f"{pf_is.trades.win_rate():.2%}" if pf_is.trades.count() > 0 else 'N/A',
        f"{pf_is.trades.profit_factor():.2f}" if pf_is.trades.count() > 0 else 'N/A',
    ],
    'Out-of-Sample': [
        f"{oos_sr:.4f}" if not np.isnan(oos_sr) else 'N/A',
        safe_metric(pf_oos, 'sortino_ratio'),
        safe_pct(pf_oos, 'total_return'), safe_pct(pf_oos, 'max_drawdown'),
        str(pf_oos.trades.count()) if pf_oos else 'N/A',
        f"{oos_wr:.2%}" if pf_oos and pf_oos.trades.count() > 0 else 'N/A',
        f"{oos_pf:.2f}" if pf_oos and pf_oos.trades.count() > 0 else 'N/A',
    ],
})

print(f"\n📊 IS vs OOS COMPARISON — {BEST_MODE} Ensemble")
print("="*75)
print(comparison.to_string(index=False))
print(f"\n{'='*75}")
if not np.isnan(degrad):
    tag = '✅ Healthy' if abs(degrad) < 30 else ('⚠️ Moderate' if abs(degrad) < 50 else '❌ Severe overfit!')
    print(f"   Sharpe Degradation: {degrad:.1f}% {tag}")
else:
    print("   Sharpe Degradation: N/A (OOS failed)")


In [ ]:
# Plot IS vs OOS side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'{BEST_MODE} Ensemble — IS vs OOS Equity Curves', fontsize=13, fontweight='bold')

pf_is.value().plot(ax=axes[0], color='navy', linewidth=1.5)
axes[0].set_title(f'In-Sample | Sharpe: {is_sr:.3f}')
axes[0].set_ylabel('Portfolio Value ($)'); axes[0].grid(alpha=0.3)

if pf_oos:
    pf_oos.value().plot(ax=axes[1], color='darkgreen', linewidth=1.5)
    axes[1].set_title(f'Out-of-Sample | Sharpe: {oos_sr:.3f}')
else:
    axes[1].text(0.5, 0.5, 'OOS: Insufficient signals', ha='center', va='center', fontsize=14)
    axes[1].set_title('Out-of-Sample | N/A')
axes[1].set_ylabel('Portfolio Value ($)'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 13. All Voting Modes — IS vs OOS Comparison

### What: Run ALL three voting modes on BOTH IS and OOS to see which generalizes best.
### Why: The "best IS mode" might not be the "best OOS mode". If OR wins IS but AND wins OOS, the market is rewarding selectivity — useful information for live deployment.
### Assumption: All params frozen. Only mode varies.


In [ ]:
print("📊 FULL VOTING MODE COMPARISON\n")
print(f"{'Mode':<10} {'IS Sharpe':>10} {'OOS Sharpe':>11} {'IS Return':>10} {'OOS Return':>11} "
      f"{'IS DD':>8} {'OOS DD':>8} {'IS Trades':>10} {'OOS Trades':>11} {'Degradation':>12}")
print("─"*110)

all_mode_comparison = []
for mode in ENSEMBLE_MODES:
    # IS
    pf_m_is = run_ensemble_backtest(train_data, BEST_E1, BEST_E2, BEST_E3,
                                     BEST_MF, BEST_MS, BEST_MSG,
                                     BEST_RL, BEST_OS, BEST_OB, mode)
    # OOS
    pf_m_oos = run_ensemble_backtest(test_data, BEST_E1, BEST_E2, BEST_E3,
                                      BEST_MF, BEST_MS, BEST_MSG,
                                      BEST_RL, BEST_OS, BEST_OB, mode)
    
    is_s = float(pf_m_is.sharpe_ratio()) if pf_m_is else float('nan')
    oos_s = float(pf_m_oos.sharpe_ratio()) if pf_m_oos else float('nan')
    is_r = float(pf_m_is.total_return()) if pf_m_is else float('nan')
    oos_r = float(pf_m_oos.total_return()) if pf_m_oos else float('nan')
    is_d = float(pf_m_is.max_drawdown()) if pf_m_is else float('nan')
    oos_d = float(pf_m_oos.max_drawdown()) if pf_m_oos else float('nan')
    is_t = int(pf_m_is.trades.count()) if pf_m_is else 0
    oos_t = int(pf_m_oos.trades.count()) if pf_m_oos else 0
    deg = (1 - oos_s/is_s)*100 if (is_s != 0 and not np.isnan(oos_s)) else float('nan')
    
    marker = ' 🏆' if mode == BEST_MODE else ''
    print(f"{mode+marker:<10} {is_s:>10.4f} {oos_s:>11.4f} {is_r:>10.2%} {oos_r:>11.2%} "
          f"{is_d:>8.2%} {oos_d:>8.2%} {is_t:>10} {oos_t:>11} {deg:>11.1f}%")
    
    all_mode_comparison.append({
        'mode': mode, 'is_sharpe': is_s, 'oos_sharpe': oos_s,
        'is_ret': is_r, 'oos_ret': oos_r, 'is_dd': is_d, 'oos_dd': oos_d,
        'is_trades': is_t, 'oos_trades': oos_t, 'degradation': deg,
        'pf_is': pf_m_is, 'pf_oos': pf_m_oos,
    })

# Identify best OOS mode
best_oos_mode = min(all_mode_comparison, key=lambda x: abs(x['degradation']) if not np.isnan(x['degradation']) else 999)
print(f"\n   Best IS Sharpe mode:  {BEST_MODE}")
print(f"   Lowest degradation:   {best_oos_mode['mode']} ({best_oos_mode['degradation']:.1f}%)")
if best_oos_mode['mode'] != BEST_MODE:
    print(f"   ⚠️  Consider using {best_oos_mode['mode']} mode for live trading (better generalization)")


## 14. Individual Indicators vs Ensemble — Head-to-Head

### What: Compare each standalone indicator against the ensemble to prove (or disprove) ensemble value.
### Why: If the ensemble doesn't beat at least 2 of 3 standalone indicators on OOS, it's adding complexity without value.
### Assumption: Uses the best params for each indicator from Phase 1.


In [ ]:
print("📊 INDIVIDUAL vs ENSEMBLE — OOS Head-to-Head\n")

# Run standalone indicators on OOS
standalone_results = {}

# TEMA alone
pf_tema_oos = None
try:
    t_ent, t_exi = compute_tema_signals(test_data['close'], BEST_E1, BEST_E2, BEST_E3)
    t_ent = t_ent & test_data['regime'] & test_data['vol_ok']
    t_ent = t_ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
    t_exi = t_exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
    if t_ent.sum() >= 3:
        pf_tema_oos = vbt.Portfolio.from_signals(test_data['close'], t_ent, t_exi, init_cash=INIT_CASH, fees=FEES, freq='D')
except: pass

# MACD-V alone
pf_macdv_oos = None
try:
    m_ent, m_exi = compute_macdv_signals(test_data['close'], test_data['atr'], BEST_MF, BEST_MS, BEST_MSG)
    m_ent = m_ent & test_data['regime'] & test_data['vol_ok']
    m_ent = m_ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
    m_exi = m_exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
    if m_ent.sum() >= 3:
        pf_macdv_oos = vbt.Portfolio.from_signals(test_data['close'], m_ent, m_exi, init_cash=INIT_CASH, fees=FEES, freq='D')
except: pass

# RSI alone
pf_rsi_oos = None
try:
    r_ent, r_exi = compute_rsi_signals(test_data['close'], BEST_RL, BEST_OS, BEST_OB)
    r_ent = r_ent & test_data['regime'] & test_data['vol_ok']
    r_ent = r_ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
    r_exi = r_exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
    if r_ent.sum() >= 3:
        pf_rsi_oos = vbt.Portfolio.from_signals(test_data['close'], r_ent, r_exi, init_cash=INIT_CASH, fees=FEES, freq='D')
except: pass

# Print comparison
def fmt(pf, metric):
    if pf is None: return 'N/A'
    try:
        v = getattr(pf, metric)()
        return f"{v:.4f}" if abs(v) < 10 else f"{v:.2%}"
    except: return 'N/A'

print(f"{'Strategy':<25} {'Sharpe':>10} {'Return':>10} {'MaxDD':>10} {'Trades':>8} {'WinRate':>10}")
print("─"*75)

for name, pf_obj in [('TEMA standalone', pf_tema_oos), ('MACD-V standalone', pf_macdv_oos), 
                       ('RSI standalone', pf_rsi_oos), (f'ENSEMBLE [{BEST_MODE}]', pf_oos)]:
    if pf_obj is not None:
        sr = f"{pf_obj.sharpe_ratio():.4f}"
        rt = f"{pf_obj.total_return():.2%}"
        dd = f"{pf_obj.max_drawdown():.2%}"
        tr = str(pf_obj.trades.count())
        wr = f"{pf_obj.trades.win_rate():.1%}" if pf_obj.trades.count() > 0 else 'N/A'
    else:
        sr = rt = dd = wr = 'N/A'; tr = '0'
    marker = ' 🏆' if name.startswith('ENSEMBLE') else ''
    print(f"{name+marker:<25} {sr:>10} {rt:>10} {dd:>10} {tr:>8} {wr:>10}")

# Count how many standalone indicators the ensemble beats
if pf_oos:
    ens_sr = float(pf_oos.sharpe_ratio())
    beaten = 0
    for pf_s in [pf_tema_oos, pf_macdv_oos, pf_rsi_oos]:
        if pf_s and float(pf_s.sharpe_ratio()) < ens_sr:
            beaten += 1
    print(f"\n   Ensemble beats {beaten}/3 standalone indicators on OOS Sharpe")
    if beaten >= 2:
        print("   ✅ Ensemble adds value — diversification is working")
    else:
        print("   ⚠️  Ensemble may not be adding value in this regime")


## 15. Sensitivity Analysis — Parameter Robustness

### What: Vary key parameters from each indicator independently around the optimum and measure Sharpe degradation on both IS and OOS.
### Why: If Sharpe collapses when RSI oversold moves from 30 to 35, you're sitting on a fragile spike. We want a plateau — a "neighborhood of good" parameters.
### Assumption: We vary one parameter at a time while holding all others fixed. This is computationally tractable but doesn't capture interaction effects.

> **What to look for:** Green bars (plateau) = robust. Red spike at optimum = fragile. OOS mirrors IS shape = generalizable.


In [ ]:
def sensitivity_sweep(param_name, values, base_params_fn, data_is, data_oos):
    """Sweep one parameter and return IS/OOS Sharpe for each value."""
    rows = []
    for val in values:
        for label, data in [('IS', data_is), ('OOS', data_oos)]:
            try:
                pf = base_params_fn(data, val)
                sr = float(pf.sharpe_ratio()) if pf else np.nan
            except:
                sr = np.nan
            rows.append({'param': param_name, 'value': val, 'split': label, 'sharpe': sr})
    return pd.DataFrame(rows)

# Define sweep functions for each key parameter
def sweep_ema1(data, val):
    if val >= BEST_E2: return None
    return run_ensemble_backtest(data, val, BEST_E2, BEST_E3, BEST_MF, BEST_MS, BEST_MSG, BEST_RL, BEST_OS, BEST_OB, BEST_MODE)

def sweep_ema3(data, val):
    if val <= BEST_E2: return None
    return run_ensemble_backtest(data, BEST_E1, BEST_E2, val, BEST_MF, BEST_MS, BEST_MSG, BEST_RL, BEST_OS, BEST_OB, BEST_MODE)

def sweep_macdv_fast(data, val):
    if val >= BEST_MS: return None
    return run_ensemble_backtest(data, BEST_E1, BEST_E2, BEST_E3, val, BEST_MS, BEST_MSG, BEST_RL, BEST_OS, BEST_OB, BEST_MODE)

def sweep_rsi_len(data, val):
    return run_ensemble_backtest(data, BEST_E1, BEST_E2, BEST_E3, BEST_MF, BEST_MS, BEST_MSG, val, BEST_OS, BEST_OB, BEST_MODE)

def sweep_rsi_oversold(data, val):
    if val >= BEST_OB: return None
    return run_ensemble_backtest(data, BEST_E1, BEST_E2, BEST_E3, BEST_MF, BEST_MS, BEST_MSG, BEST_RL, val, BEST_OB, BEST_MODE)

def sweep_rsi_overbought(data, val):
    if val <= BEST_OS: return None
    return run_ensemble_backtest(data, BEST_E1, BEST_E2, BEST_E3, BEST_MF, BEST_MS, BEST_MSG, BEST_RL, BEST_OS, val, BEST_MODE)

# Run sweeps
N = SENSITIVITY_STEPS
sweeps = [
    ('EMA1 (fast)', [BEST_E1 + d*2 for d in range(-N, N+1) if BEST_E1 + d*2 >= 3], sweep_ema1),
    ('EMA3 (slow)', [BEST_E3 + d*10 for d in range(-N, N+1) if BEST_E3 + d*10 >= 20], sweep_ema3),
    ('MACD Fast',   [BEST_MF + d*2 for d in range(-N, N+1) if BEST_MF + d*2 >= 4], sweep_macdv_fast),
    ('RSI Length',  [BEST_RL + d*2 for d in range(-N, N+1) if BEST_RL + d*2 >= 4], sweep_rsi_len),
    ('RSI Oversold',[BEST_OS + d*5 for d in range(-N, N+1) if 10 <= BEST_OS + d*5 <= 45], sweep_rsi_oversold),
    ('RSI Overbought',[BEST_OB + d*5 for d in range(-N, N+1) if 55 <= BEST_OB + d*5 <= 90], sweep_rsi_overbought),
]

all_sens = []
print("🔄 Running sensitivity sweeps...\n")
for name, vals, fn in sweeps:
    sdf = sensitivity_sweep(name, vals, fn, train_data, test_data)
    all_sens.append(sdf)
    n_valid = sdf['sharpe'].notna().sum()
    print(f"   {name:<18}: {len(vals)} values, {n_valid} valid results")

sens_all = pd.concat(all_sens, ignore_index=True)
print(f"\n✅ Sensitivity complete: {len(sens_all)} data points")


In [ ]:
# Plot sensitivity charts
param_names = sens_all['param'].unique()
n_params = len(param_names)
fig, axes = plt.subplots(2, n_params, figsize=(4*n_params, 10))
fig.suptitle(f'Sensitivity Analysis — {BEST_MODE} Ensemble', fontsize=14, fontweight='bold')

base_is_sharpe = float(pf_is.sharpe_ratio())
base_oos_sharpe = float(pf_oos.sharpe_ratio()) if pf_oos else 0

for col, pname in enumerate(param_names):
    subset = sens_all[sens_all['param'] == pname]
    
    for row, split_label in enumerate(['IS', 'OOS']):
        ax = axes[row, col] if n_params > 1 else axes[row]
        split_data = subset[subset['split'] == split_label].sort_values('value')
        
        if len(split_data) == 0: continue
        
        base_sr = base_is_sharpe if split_label == 'IS' else base_oos_sharpe
        if base_sr == 0: base_sr = 0.001
        
        deltas = ((split_data['sharpe'] - base_sr) / abs(base_sr) * 100).fillna(0)
        colors = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' for x in deltas]
        
        ax.bar(split_data['value'], deltas, color=colors, edgecolor='black', linewidth=0.5)
        ax.axhline(0, color='black', linewidth=1.5)
        ax.set_title(f'{pname} — {split_label}', fontweight='bold', fontsize=10)
        ax.set_ylabel('Sharpe Δ%')
        ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("\n📋 SENSITIVITY SUMMARY")
print("="*80)
for pname in param_names:
    subset = sens_all[(sens_all['param'] == pname) & (sens_all['split'] == 'IS')]
    is_vals = subset['sharpe'].dropna()
    if len(is_vals) == 0: continue
    max_drop = ((is_vals.min() - base_is_sharpe) / abs(base_is_sharpe) * 100) if base_is_sharpe != 0 else 0
    tag = '⚠️ HIGH' if abs(max_drop) > 40 else '✅ LOW'
    print(f"   {pname:<18}: IS range [{is_vals.min():.3f}, {is_vals.max():.3f}] | Max drop: {max_drop:.1f}% | {tag}")


## 16. Signal Agreement Analysis — How Often Do Indicators Agree?

### What: Measure the correlation and agreement rate between the three indicators' signals.
### Why: Per Carver, the diversification multiplier depends on inter-rule correlation. Low correlation (~0.25) means the ensemble provides true diversification. High correlation means you're just paying complexity tax for the same signal three times.
### Assumption: Using full dataset signals (for visualization only — not affecting optimization).


In [ ]:
# Generate signals on full training data
t_ent_full, _ = compute_tema_signals(train_data['close'], BEST_E1, BEST_E2, BEST_E3)
m_ent_full, _ = compute_macdv_signals(train_data['close'], train_data['atr'], BEST_MF, BEST_MS, BEST_MSG)
r_ent_full, _ = compute_rsi_signals(train_data['close'], BEST_RL, BEST_OS, BEST_OB)

# Convert to continuous position signals (1 if "would be long", 0 otherwise)
def to_position(entries, exits):
    pos = pd.Series(0, index=entries.index)
    in_trade = False
    for i in range(len(entries)):
        if entries.iloc[i]: in_trade = True
        if exits.iloc[i]: in_trade = False
        pos.iloc[i] = 1 if in_trade else 0
    return pos

# For speed, use a vectorized approximation of position state
ent_df = pd.DataFrame({
    'TEMA': t_ent_full.astype(int),
    'MACD-V': m_ent_full.astype(int),
    'RSI': r_ent_full.astype(int),
}).fillna(0)

# Correlation matrix of entry signals
corr_matrix = ent_df.corr()
print("📊 Entry Signal Correlation Matrix (training data)")
print("="*50)
print(corr_matrix.round(3).to_string())

# Agreement rates
total_bars = len(ent_df)
all_agree = ((ent_df.sum(axis=1) == 3) | (ent_df.sum(axis=1) == 0)).sum()
two_agree = (ent_df.sum(axis=1) >= 2).sum()
any_fire  = (ent_df.sum(axis=1) >= 1).sum()

print(f"\n📊 Signal Agreement Rates")
print(f"   All 3 agree (entry or no entry): {all_agree/total_bars:.1%}")
print(f"   At least 2 fire simultaneously:  {two_agree/total_bars:.1%}")
print(f"   At least 1 fires:                {any_fire/total_bars:.1%}")

avg_corr = (corr_matrix.values[np.triu_indices(3, k=1)]).mean()
print(f"\n   Average pairwise correlation: {avg_corr:.3f}")

# Carver's diversification multiplier estimate
if avg_corr >= 0:
    div_mult = 1 / np.sqrt(1/3 + 2/3 * avg_corr)
    print(f"   Estimated Carver diversification multiplier: {min(div_mult, 2.5):.2f} (capped at 2.5)")
    if avg_corr < 0.3:
        print("   ✅ Low correlation — genuine diversification benefit")
    elif avg_corr < 0.5:
        print("   ⚠️  Moderate correlation — some diversification")
    else:
        print("   ❌ High correlation — limited ensemble benefit")


## 17. Final Audit — Machine Learning Ridicule

### What: A ruthless self-audit identifying everything that could have "worked by accident."
### Why: Per Meta Prompt 1: *"Assume the author is competent but possibly overconfident."*

This cell exists to make you uncomfortable. If nothing here concerns you, you didn't read it.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    🔍 FINAL AUDIT — MACHINE LEARNING RIDICULE              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  1. TWO-PHASE OPTIMIZATION IS NOT INNOCENT                                 ║
║     We optimized each indicator separately, THEN picked the voting mode.   ║
║     This is better than joint optimization but still has hidden degrees     ║
║     of freedom: 3 indicator grids + 3 voting modes = multiple testing.     ║
║     Bonferroni correction would shrink significance considerably.          ║
║     Risk: SERIOUS                                                          ║
║                                                                            ║
║  2. RSI IN AN ENSEMBLE IS A CONTRADICTION                                  ║
║     RSI is mean-reversion. TEMA and MACD-V are trend-following.            ║
║     In AND mode, requiring both trend AND mean-reversion signals           ║
║     simultaneously may produce entries only at very specific points         ║
║     (trend resumes after pullback) — which may or may not recur.           ║
║     Risk: MODERATE (but feature, not bug, if conscious)                    ║
║                                                                            ║
║  3. EXIT LOGIC ASYMMETRY                                                   ║
║     Entry logic varies by mode but exit logic is biased toward             ║
║     early exits (safety). This WILL reduce returns in strong trends.       ║
║     Consider testing symmetric exit modes too.                             ║
║     Risk: MODERATE                                                         ║
║                                                                            ║
║  4. SURVIVORSHIP BIAS — SINGLE ASSET                                       ║
║     BTC-USD is the biggest crypto winner. Test on ETH, SOL, and            ║
║     especially FAILED assets before deploying real capital.                ║
║     Risk: CRITICAL                                                         ║
║                                                                            ║
║  5. REGIME SAMPLE SIZE                                                     ║
║     2018-2025 BTC = 1 bear (2018), 1 bull (2020-21), 1 bear (2022),      ║
║     1 bull (2023-25). That's N=4 regime transitions. Not a sample.        ║
║     Risk: CRITICAL                                                         ║
║                                                                            ║
║  6. CARVER'S WARNING ON DIVERSIFICATION MULTIPLIER                         ║
║     "Beware of using very high multipliers." If entry signal               ║
║     correlation is near zero, the div multiplier inflates position         ║
║     conviction — which is dangerous if the low correlation is              ║
║     accidental rather than structural.                                     ║
║     Risk: MODERATE                                                         ║
║                                                                            ║
║  7. FEES ARE STILL OPTIMISTIC                                              ║
║     0.1% flat ignores: spreads widening during vol, overnight funding,    ║
║     FTMO-specific commission structures, and execution timing.            ║
║     Risk: SERIOUS                                                          ║
║                                                                            ║
║  8. NO POSITION SIZING OR RISK MANAGEMENT                                  ║
║     Binary 100% in/out. No vol targeting, no max drawdown stop,           ║
║     no Kelly criterion, no FTMO 5%/10% drawdown limits.                   ║
║     Risk: CRITICAL for live trading                                        ║
║                                                                            ║
║  9. SENSITIVITY ANALYSIS DOESN'T CAPTURE INTERACTIONS                      ║
║     We varied one param at a time. In reality, EMA1 and RSI oversold      ║
║     interact — a fast EMA with a tight oversold may behave very           ║
║     differently than the sum of their individual sensitivities.            ║
║     Risk: MODERATE                                                         ║
║                                                                            ║
║  10. "WORKED BY ACCIDENT" CHECKLIST                                        ║
║      □ Best param at grid boundary? → Boundary optimum = overfitting      ║
║      □ OOS Sharpe > IS Sharpe? → Suspect data leak                       ║
║      □ < 30 trades in any split? → Sharpe estimate unreliable             ║
║      □ One giant trade dominates returns? → Lucky, not skilled             ║
║      □ Ensemble worse than all standalone indicators? → No value           ║
║                                                                            ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  BEFORE DEPLOYING:                                                         ║
║  1. Test on 3+ additional assets                                           ║
║  2. Add volatility-targeted position sizing (Carver Ch. 10)                ║
║  3. Implement FTMO drawdown limits                                         ║
║  4. Paper trade for 3+ months                                              ║
║  5. Run walk-forward validation                                            ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# Automated boundary checks
print("🤖 AUTOMATED CHECKS:")
for name, val, rng in [('EMA1', BEST_E1, list(EMA1_RANGE)), ('EMA3', BEST_E3, list(EMA3_RANGE)),
                        ('MACD Fast', BEST_MF, list(MACDV_FAST)), ('RSI Len', BEST_RL, list(RSI_LEN_RANGE)),
                        ('RSI OS', BEST_OS, list(RSI_OVERSOLD_RANGE)), ('RSI OB', BEST_OB, list(RSI_OVERBOUGHT_RANGE))]:
    at_boundary = val in [rng[0], rng[-1]] if len(rng) > 1 else False
    print(f"   {name:>12} = {val} {'⚠️ AT BOUNDARY' if at_boundary else '✅ Interior'}")

if pf_oos:
    oos_val = float(pf_oos.sharpe_ratio())
    is_val = float(pf_is.sharpe_ratio())
    if oos_val > is_val and is_val > 0:
        print(f"   ❌ OOS Sharpe ({oos_val:.4f}) > IS Sharpe ({is_val:.4f}) — SUSPICIOUS!")
    if pf_is.trades.count() < 30:
        print(f"   ⚠️  IS trades: {pf_is.trades.count()} (< 30 — unreliable Sharpe)")
    if pf_oos.trades.count() < 15:
        print(f"   ⚠️  OOS trades: {pf_oos.trades.count()} (< 15 — insufficient validation)")

print(f"\n✅ Audit complete. Read everything above before risking capital.")


---
## End of Notebook

**Configuration recap:**
- Asset: configured in Section 2
- Ensemble: TEMA + MACD-V + RSI with AND/MAJORITY/OR voting
- Best mode selected automatically by IS Sharpe, validated on OOS
- Sensitivity analysis covers 6 key parameters across both splits

**Next steps:**
1. Run all cells top-to-bottom — never skip
2. If OOS degrades >50%, try loosening the ensemble (switch from AND → MAJORITY)
3. If RSI is dragging the ensemble, try widening oversold/overbought thresholds
4. Test on additional assets (ETH-USD, EURUSD=X, ^GSPC)
5. Add position sizing before going live

*"Combining rules from different styles reduces correlation to ~0.25 — true diversification."* — Robert Carver
